# Rock Paper Scissors - Klasifikasi Gambar dengan CNN

## Langkah 1: Unduh dan Ekstrak Dataset

Unduh dataset Rock Paper Scissors, lalu ekstrak sehingga struktur foldernya menjadi:
```
├── main.ipynb
rockpaperscissors/
    ├── paper/
    │   ├── image 1.png
    │   ├── image 2.png
    │   └── ...
    ├── rock/
    │   ├── image 1.png
    │   ├── image 2.png
    │   └── ...
    └── scissors/
        ├── image 1.png
        ├── image 2.png
        └── ...
```

## Langkah 2: Import Library

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Conv2D, MaxPooling2D
from tensorflow.keras.preprocessing.image import ImageDataGenerator

## Langkah 3: Persiapan Data dengan ImageDataGenerator

ImageDataGenerator digunakan untuk mempersiapkan data latih dan validasi. Kemudahan yang disediakan antara lain:
- Preprocessing data (rescaling)
- Pelabelan sampel otomatis berdasarkan nama folder
- Augmentasi gambar

In [ ]:
dataset_path = "./rockpaperscissors"

train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_generator = train_datagen.flow_from_directory(
    dataset_path,
    target_size=(150, 150),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

validation_generator = train_datagen.flow_from_directory(
    dataset_path,
    target_size=(150, 150),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

## Langkah 4: Membangun Arsitektur Model CNN

Model CNN terdiri dari:
- **Layer Konvolusi (Conv2D)**: Mengekstraksi fitur penting dari gambar
- **Layer Max Pooling**: Mengurangi resolusi gambar untuk mempercepat pelatihan
- **Flatten**: Mengubah matriks 2D menjadi vektor 1D
- **Dense**: Layer fully connected untuk klasifikasi akhir

CNN lebih optimal dibanding MLP karena fokus pada fitur unik yang membedakan tiap kategori, bukan keseluruhan piksel gambar.

In [ ]:
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(150, 150, 3)),
    MaxPooling2D(2, 2),

    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),

    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),

    Flatten(),
    Dense(512, activation='relu'),
    Dense(3, activation='softmax')
])

model.summary()

## Langkah 5: Kompilasi Model

- **Loss function**: `categorical_crossentropy` → digunakan untuk klasifikasi multi-kelas
- **Optimizer**: `adam` → optimizer adaptif yang efisien
- **Metrics**: `accuracy` → untuk memantau akurasi selama pelatihan

In [ ]:
model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

## Langkah 6: Pelatihan Model (Model Fitting)

Proses pelatihan menggunakan `ImageDataGenerator` sehingga gambar dan labelnya dimasukkan secara otomatis. Gambar di folder `rock` otomatis dilabeli `rock`, dan seterusnya.

In [ ]:
history = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=10
)

## Langkah 7: Evaluasi Model

In [ ]:
val_loss, val_acc = model.evaluate(validation_generator)
print(f'Validation loss: {val_loss}, Validation accuracy: {val_acc}')

## Langkah 8: Prediksi pada Data Validasi

In [ ]:
predictions = model.predict(validation_generator)
print(predictions)  # Output berupa probabilitas prediksi tiap kelas